What this pipeline does

It will:

install tools
read and clean quantitative.xlsx
arrange phenotype data correctly
convert album_final_noF.vcf.gz to PLINK
filter genotypes lightly
install GEMMA
compute kinship once
run one GWAS per quantitative trait
save results, top hits, summary tables, Manhattan plots, and QQ plots

1) Expected input files

Put these in your Colab working directory:

album_final_noF.vcf.gz
quantitative.xlsx

Optional later:

qualitative.xlsx
seed character both qualitative and quantitative.xlsx

Your notebook confirms quantitative.xlsx contains these quantitative traits: Plant Height, Stem Breadth, Leaf Length, Leaf breadth, Petiole length, Petiole breadth, and Panicle length.


1) Expected input files

Put these in your Colab working directory:

album_final_noF.vcf.gz
quantitative.xlsx

Optional later:

qualitative.xlsx
seed character both qualitative and quantitative.xlsx

Your notebook confirms quantitative.xlsx contains these quantitative traits: Plant Height, Stem Breadth, Leaf Length, Leaf breadth, Petiole length, Petiole breadth, and Panicle length.

2) Phenotype arrangement you should use

Your quantitative phenotype sheet should effectively look like this structure:

Trait name              Sample_A   Sample_B   Sample_C   Sample_D   Sample_E
Plant Height (m)        0.6±0.05   0.6±0.05   2.1±0.05   0.8±0.2    1.1±0.11
Stem Breadth (cm)       0.9±0.2    0.8±0.2    5.3±0.9    0.6±0.1    1.0±0.06
Leaf Length (cm)        5.6±0.2    2.5±0.1    7.3±0.2    8.6±0.1    5.8±0.23
...

After cleaning and transposing, GEMMA should receive it as:

sample_1   trait1 trait2 trait3 ...
sample_2   trait1 trait2 trait3 ...
sample_3   trait1 trait2 trait3 ...
sample_4   trait1 trait2 trait3 ...
sample_5   trait1 trait2 trait3 ...

Your notebook already produced exactly that cleaned 5-row trait matrix.

In [1]:
!apt-get update -qq
!apt-get install -y plink1.9 wget

!pip install -q pandas numpy matplotlib openpyxl

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
wget is already the newest version (1.21.2-2ubuntu1.1).
The following NEW packages will be installed:
  plink1.9
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 865 kB of archives.
After this operation, 2,170 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 plink1.9 amd64 1.90~b6.21-201019-1 [865 kB]
Fetched 865 kB in 1s (1,320 kB/s)
Selecting previously unselected package plink1.9.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../plink1.9_1.90~b6.21-201019-1_amd64.deb ...
Unpacking plink1.9 (1.90~b6.21-201019-1) ...
Setting up plink1.9 (1.90~b6.21-201019-1) ...
Processing trigg

# Read and arrange phenotype data from quantitative.xlsx

In [2]:
import pandas as pd
import numpy as np

# Read raw Excel exactly as in your working notebook style
df = pd.read_excel("quantitative.xlsx", header=None)

# First column = trait names
traits = df.iloc[:, 0].astype(str)

# Remaining columns = sample values
data = df.iloc[:, 1:].T
data.columns = traits
data.reset_index(drop=True, inplace=True)

def extract_mean(x):
    """
    Convert values like '0.6 ±0.05' or '1.1± 0.11' to 0.6 and 1.1
    """
    try:
        if isinstance(x, str):
            x = x.replace("± ", "±").replace(" ±", "±").strip()
            return float(x.split("±")[0].strip())
        elif isinstance(x, (int, float, np.integer, np.floating)):
            return float(x)
    except Exception:
        return np.nan
    return np.nan

clean_df = data.apply(lambda col: col.map(extract_mean))
clean_df = clean_df.dropna(axis=1, how="all")

print("Clean quantitative phenotype matrix:")
print(clean_df)
print("\nShape:", clean_df.shape)
print("\nTraits:")
print(list(clean_df.columns))

Clean quantitative phenotype matrix:
0  Plant Height (m)  Stem Breadth (cm)  Leaf Length (cm)  Leaf breadth (cm)  \
0               0.6                0.9               5.6                3.7   
1               0.6                0.8               2.5                0.6   
2               2.1                5.3               7.3                3.4   
3               0.8                0.6               8.6                4.9   
4               1.1                1.0               5.8                3.0   

0  Petiole length (cm)  Petiole breadth (mm)  Panicle length (cm)  
0                  5.7                   1.7                 13.3  
1                  1.3                   0.4                 23.0  
2                  2.6                   1.0                 23.7  
3                  6.7                   1.6                 18.0  
4                  3.5                   1.0                 21.0  

Shape: (5, 7)

Traits:
['Plant Height (m)', 'Stem Breadth (cm)', 'Leaf Length (

Convert VCF to PLINK

In [7]:
!plink1.9 --bfile geno \
          --geno 0.2 \
          --make-bed \
          --allow-extra-chr \
          --out geno_filtered

PLINK v1.90b6.22 64-bit (3 Nov 2020)           www.cog-genomics.org/plink/1.9/
(C) 2005-2020 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to geno_filtered.log.
Options in effect:
  --allow-extra-chr
  --bfile geno
  --geno 0.2
  --make-bed
  --out geno_filtered

12975 MB RAM detected; reserving 6487 MB for main workspace.
702016 variants loaded from .bim file.
5 people (0 males, 0 females, 5 ambiguous) loaded from .fam.
Ambiguous sex IDs written to geno_filtered.nosex .
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 5 founders and 0 nonfounders present.
Calculating allele frequencies... 0%1%2%3%4%5%6%7%8%9%10%11%12%13%14%15%16%17%18%19%20%21%22%23%24%25%26%27%28%29%30%31%32%33%34%35%36%37%38%39%40%41%42%43%44%45%46%47%48%49%50%51%52%53%54%55%56%57%58%59%

In [8]:
import pandas as pd

fam = pd.read_csv("geno_filtered.fam", sep=r"\s+", header=None)
fam.columns = ["FID","IID","PID","MID","SEX","PHENO"]

print(fam)

                            FID                           IID  PID  MID  SEX  \
0  bam_album/Album_A.sorted.bam  bam_album/Album_A.sorted.bam    0    0    0   
1  bam_album/Album_B.sorted.bam  bam_album/Album_B.sorted.bam    0    0    0   
2  bam_album/Album_C.sorted.bam  bam_album/Album_C.sorted.bam    0    0    0   
3  bam_album/Album_D.sorted.bam  bam_album/Album_D.sorted.bam    0    0    0   
4  bam_album/Album_E.sorted.bam  bam_album/Album_E.sorted.bam    0    0    0   

   PHENO  
0     -9  
1     -9  
2     -9  
3     -9  
4     -9  


In [11]:
!mkdir -p output

!./gemma -bfile geno_filtered \
         -gk 1 \
         -o kinship

/bin/bash: line 1: ./gemma: No such file or directory


In [12]:
!ls -lh

total 51M
-rw-r--r-- 1 root root  11M Mar 25 05:28  album_final_noF.vcf.gz
-rw-r--r-- 1 root root 431K Mar 25 05:24  album_final_noF.vcf.gz.csi
-rw-r--r-- 1 root root 1.4M Mar 25 05:28  geno.bed
-rw-r--r-- 1 root root  19M Mar 25 05:28  geno.bim
-rw-r--r-- 1 root root  335 Mar 25 05:28  geno.fam
-rw-r--r-- 1 root root 1.4M Mar 25 05:29  geno_filtered.bed
-rw-r--r-- 1 root root  19M Mar 25 05:29  geno_filtered.bim
-rw-r--r-- 1 root root  335 Mar 25 05:29  geno_filtered.fam
-rw-r--r-- 1 root root  925 Mar 25 05:29  geno_filtered.log
-rw-r--r-- 1 root root  290 Mar 25 05:29  geno_filtered.nosex
-rw-r--r-- 1 root root  916 Mar 25 05:28  geno.log
-rw-r--r-- 1 root root  290 Mar 25 05:28  geno.nosex
drwxr-xr-x 2 root root 4.0K Mar 25 05:32  output
-rw-r--r-- 1 root root  12K Mar 25 05:24  qualitative.xlsx
-rw-r--r-- 1 root root 9.7K Mar 25 05:24  quantitative.xlsx
drwxr-xr-x 1 root root 4.0K Mar 23 13:34  sample_data
-rw-r--r-- 1 root root 9.6K Mar 25 05:24 'seed character both qualitative a

In [13]:
!rm -f gemma gemma-0.98.5-linux-static-AMD64 gemma-0.98.5-linux-static-AMD64.gz

!wget -O gemma-0.98.5-linux-static-AMD64.gz \
  https://github.com/genetics-statistics/GEMMA/releases/download/v0.98.5/gemma-0.98.5-linux-static-AMD64.gz

!gunzip -f gemma-0.98.5-linux-static-AMD64.gz
!chmod +x gemma-0.98.5-linux-static-AMD64
!mv gemma-0.98.5-linux-static-AMD64 gemma

!ls -lh gemma
!./gemma -h | head -20

--2026-03-25 05:32:34--  https://github.com/genetics-statistics/GEMMA/releases/download/v0.98.5/gemma-0.98.5-linux-static-AMD64.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/2880217/634e9bdc-e9f8-409d-8354-16874abd27ce?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-25T06%3A28%3A56Z&rscd=attachment%3B+filename%3Dgemma-0.98.5-linux-static-AMD64.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-25T05%3A28%3A49Z&ske=2026-03-25T06%3A28%3A56Z&sks=b&skv=2018-11-09&sig=A9cwZsp%2BS%2BX%2BT2skyTb71ui63gx65FaHuvJ1ewT46a4%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NDQxNzA1NCwibmJmIjoxNzc0NDE2N

In [15]:
import pandas as pd
import numpy as np

df = pd.read_excel("quantitative.xlsx", header=None)

traits = df.iloc[:, 0].astype(str)
data = df.iloc[:, 1:].T
data.columns = traits
data.reset_index(drop=True, inplace=True)

def extract_mean(x):
    try:
        if isinstance(x, str):
            x = x.replace("± ", "±").replace(" ±", "±").strip()
            return float(x.split("±")[0].strip())
        elif isinstance(x, (int, float, np.integer, np.floating)):
            return float(x)
    except:
        return np.nan
    return np.nan

clean_df = data.apply(lambda col: col.map(extract_mean))
clean_df = clean_df.dropna(axis=1, how="all")

print(clean_df)
print(clean_df.columns.tolist())

0  Plant Height (m)  Stem Breadth (cm)  Leaf Length (cm)  Leaf breadth (cm)  \
0               0.6                0.9               5.6                3.7   
1               0.6                0.8               2.5                0.6   
2               2.1                5.3               7.3                3.4   
3               0.8                0.6               8.6                4.9   
4               1.1                1.0               5.8                3.0   

0  Petiole length (cm)  Petiole breadth (mm)  Panicle length (cm)  
0                  5.7                   1.7                 13.3  
1                  1.3                   0.4                 23.0  
2                  2.6                   1.0                 23.7  
3                  6.7                   1.6                 18.0  
4                  3.5                   1.0                 21.0  
['Plant Height (m)', 'Stem Breadth (cm)', 'Leaf Length (cm)', 'Leaf breadth (cm)', 'Petiole length (cm)', 'Petiole br

In [16]:
trait = "Plant Height (m)"
values = clean_df[trait].values

fam = pd.read_csv("geno_filtered.fam", sep=r"\s+", header=None)
fam.columns = ["FID","IID","PID","MID","SEX","PHENO"]

print("Before:")
print(fam)

fam["PHENO"] = values
fam.to_csv("geno_filtered.fam", sep="\t", index=False, header=False)

print("\nAfter:")
print(fam)

Before:
                            FID                           IID  PID  MID  SEX  \
0  bam_album/Album_A.sorted.bam  bam_album/Album_A.sorted.bam    0    0    0   
1  bam_album/Album_B.sorted.bam  bam_album/Album_B.sorted.bam    0    0    0   
2  bam_album/Album_C.sorted.bam  bam_album/Album_C.sorted.bam    0    0    0   
3  bam_album/Album_D.sorted.bam  bam_album/Album_D.sorted.bam    0    0    0   
4  bam_album/Album_E.sorted.bam  bam_album/Album_E.sorted.bam    0    0    0   

   PHENO  
0     -9  
1     -9  
2     -9  
3     -9  
4     -9  

After:
                            FID                           IID  PID  MID  SEX  \
0  bam_album/Album_A.sorted.bam  bam_album/Album_A.sorted.bam    0    0    0   
1  bam_album/Album_B.sorted.bam  bam_album/Album_B.sorted.bam    0    0    0   
2  bam_album/Album_C.sorted.bam  bam_album/Album_C.sorted.bam    0    0    0   
3  bam_album/Album_D.sorted.bam  bam_album/Album_D.sorted.bam    0    0    0   
4  bam_album/Album_E.sorted.bam  bam_

In [17]:
!mkdir -p output
!./gemma -bfile geno_filtered -gk 1 -o kinship

GEMMA 0.98.5 (2021-08-25) by Xiang Zhou, Pjotr Prins and team (C) 2012-2021
Reading Files ... 
## number of total individuals = 5
## number of analyzed individuals = 5
## number of covariates = 1
## number of phenotypes = 1
## number of total SNPs/var        =   702016
## number of analyzed SNPs         =    53307
Calculating Relatedness Matrix ... 
================================================== 100%
**** INFO: Done.


In [18]:
!./gemma -bfile geno_filtered \
         -k output/kinship.cXX.txt \
         -lmm 4 \
         -o Plant_Height_m

GEMMA 0.98.5 (2021-08-25) by Xiang Zhou, Pjotr Prins and team (C) 2012-2021
Reading Files ... 
## number of total individuals = 5
## number of analyzed individuals = 5
## number of covariates = 1
## number of phenotypes = 1
## number of total SNPs/var        =   702016
## number of analyzed SNPs         =    53307
Start Eigen-Decomposition...
pve estimate =2.4002e-06
se(pve) =0.648096

**** INFO: Done.


In [19]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt

# Read phenotype Excel
df = pd.read_excel("quantitative.xlsx", header=None)

# First column = trait names
traits = df.iloc[:, 0].astype(str)

# Remaining columns = sample values
data = df.iloc[:, 1:].T
data.columns = traits
data.reset_index(drop=True, inplace=True)

def extract_mean(x):
    """
    Convert values like '0.6 ±0.05' or '1.1± 0.11' to 0.6 and 1.1
    """
    try:
        if isinstance(x, str):
            x = x.replace("± ", "±").replace(" ±", "±").strip()
            return float(x.split("±")[0].strip())
        elif isinstance(x, (int, float, np.integer, np.floating)):
            return float(x)
    except:
        return np.nan
    return np.nan

clean_df = data.apply(lambda col: col.map(extract_mean))
clean_df = clean_df.dropna(axis=1, how="all")

print("Clean phenotype matrix:")
print(clean_df)
print("\nShape:", clean_df.shape)
print("\nTraits:")
print(list(clean_df.columns))

Clean phenotype matrix:
0  Plant Height (m)  Stem Breadth (cm)  Leaf Length (cm)  Leaf breadth (cm)  \
0               0.6                0.9               5.6                3.7   
1               0.6                0.8               2.5                0.6   
2               2.1                5.3               7.3                3.4   
3               0.8                0.6               8.6                4.9   
4               1.1                1.0               5.8                3.0   

0  Petiole length (cm)  Petiole breadth (mm)  Panicle length (cm)  
0                  5.7                   1.7                 13.3  
1                  1.3                   0.4                 23.0  
2                  2.6                   1.0                 23.7  
3                  6.7                   1.6                 18.0  
4                  3.5                   1.0                 21.0  

Shape: (5, 7)

Traits:
['Plant Height (m)', 'Stem Breadth (cm)', 'Leaf Length (cm)', 'Leaf b

In [20]:
fam = pd.read_csv("geno_filtered.fam", sep=r"\s+", header=None)
fam.columns = ["FID", "IID", "PID", "MID", "SEX", "PHENO"]

print("Genotype sample order:")
print(fam[["FID", "IID"]])

if len(fam) != len(clean_df):
    raise ValueError(f"Mismatch: geno_filtered.fam has {len(fam)} samples, but phenotype matrix has {len(clean_df)} rows.")

print("\nSample count matches phenotype rows.")

Genotype sample order:
                            FID                           IID
0  bam_album/Album_A.sorted.bam  bam_album/Album_A.sorted.bam
1  bam_album/Album_B.sorted.bam  bam_album/Album_B.sorted.bam
2  bam_album/Album_C.sorted.bam  bam_album/Album_C.sorted.bam
3  bam_album/Album_D.sorted.bam  bam_album/Album_D.sorted.bam
4  bam_album/Album_E.sorted.bam  bam_album/Album_E.sorted.bam

Sample count matches phenotype rows.


In [21]:
def safe_name(name: str) -> str:
    name = name.strip()
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"[\s/]+", "_", name)
    return name

def chr_to_num(x):
    x = str(x)
    try:
        return int(x)
    except:
        x = x.upper()
        if x == "X":
            return 23
        elif x == "Y":
            return 24
        elif x in ["MT", "M"]:
            return 25
        else:
            return 99

# Create output folders
Path("trait_pheno_files").mkdir(exist_ok=True)
Path("trait_results").mkdir(exist_ok=True)
Path("trait_plots").mkdir(exist_ok=True)

In [22]:
import subprocess

summary_rows = []

# Keep a clean copy of fam to reuse each time
fam_base = fam.copy()

for trait in clean_df.columns:
    print(f"\n===== Running GWAS for: {trait} =====")

    values = clean_df[trait].astype(float).values
    safe_trait = safe_name(trait)

    # Update PHENO column in .fam
    fam_trait = fam_base.copy()
    fam_trait["PHENO"] = values
    fam_trait.to_csv("geno_filtered.fam", sep="\t", index=False, header=False)

    # Save phenotype file for record-keeping
    pheno_record = fam_trait[["FID", "IID", "PHENO"]].copy()
    pheno_record.to_csv(
        f"trait_pheno_files/{safe_trait}.phenotype.tsv",
        sep="\t",
        index=False
    )

    # Run GEMMA
    cmd = [
        "./gemma",
        "-bfile", "geno_filtered",
        "-k", "output/kinship.cXX.txt",
        "-lmm", "4",
        "-o", safe_trait
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)
    print("Return code:", res.returncode)

    assoc_file = f"output/{safe_trait}.assoc.txt"
    log_file = f"output/{safe_trait}.log.txt"

    if res.returncode != 0 or not os.path.exists(assoc_file):
        print("GWAS failed for:", trait)
        print("STDOUT:\n", res.stdout)
        print("STDERR:\n", res.stderr)
        summary_rows.append({
            "trait": trait,
            "safe_trait": safe_trait,
            "status": "FAILED",
            "n_snps": np.nan,
            "top_p_wald": np.nan,
            "top_chr": np.nan,
            "top_ps": np.nan
        })
        continue

    # Read result
    gwas = pd.read_csv(assoc_file, sep="\t")
    gwas = gwas.dropna(subset=["p_wald"]).copy()

    if len(gwas) == 0:
        summary_rows.append({
            "trait": trait,
            "safe_trait": safe_trait,
            "status": "NO_VALID_P",
            "n_snps": 0,
            "top_p_wald": np.nan,
            "top_chr": np.nan,
            "top_ps": np.nan
        })
        continue

    gwas = gwas.sort_values("p_wald").copy()
    top = gwas.iloc[0]

    # Save top hits
    gwas.head(20).to_csv(f"trait_results/{safe_trait}_top20.csv", index=False)
    gwas.head(100).to_csv(f"trait_results/{safe_trait}_top100.csv", index=False)
    gwas.to_csv(f"trait_results/{safe_trait}_full_results.csv", index=False)

    summary_rows.append({
        "trait": trait,
        "safe_trait": safe_trait,
        "status": "OK",
        "n_snps": len(gwas),
        "top_p_wald": top["p_wald"],
        "top_chr": top["chr"],
        "top_ps": top["ps"]
    })

    print(f"Finished {trait} | top p = {top['p_wald']:.6g}")


===== Running GWAS for: Plant Height (m) =====
Return code: 0
Finished Plant Height (m) | top p = 0.00560804

===== Running GWAS for: Stem Breadth (cm) =====
Return code: 0
Finished Stem Breadth (cm) | top p = 0.000170196

===== Running GWAS for: Leaf Length (cm) =====
Return code: 0
Finished Leaf Length (cm) | top p = 0.0159083

===== Running GWAS for: Leaf breadth (cm) =====
Return code: 0
Finished Leaf breadth (cm) | top p = 0.00718954

===== Running GWAS for: Petiole length (cm) =====
Return code: 0
Finished Petiole length (cm) | top p = 0.012955

===== Running GWAS for: Petiole breadth (mm) =====
Return code: 0
Finished Petiole breadth (mm) | top p = 0.0228071

===== Running GWAS for: Panicle length (cm) =====
Return code: 0
Finished Panicle length (cm) | top p = 0.0277324


In [23]:
fam_base.to_csv("geno_filtered.fam", sep="\t", index=False, header=False)
print("Restored original geno_filtered.fam")

Restored original geno_filtered.fam


In [24]:
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(["status", "top_p_wald"], na_position="last")

print(summary_df)
summary_df.to_csv("trait_results/gwas_trait_summary.csv", index=False)
print("\nSaved: trait_results/gwas_trait_summary.csv")

                  trait          safe_trait status  n_snps  top_p_wald  \
1     Stem Breadth (cm)     Stem_Breadth_cm     OK   53307    0.000170   
0      Plant Height (m)      Plant_Height_m     OK   53307    0.005608   
3     Leaf breadth (cm)     Leaf_breadth_cm     OK   53307    0.007190   
4   Petiole length (cm)   Petiole_length_cm     OK   53307    0.012955   
2      Leaf Length (cm)      Leaf_Length_cm     OK   53307    0.015908   
5  Petiole breadth (mm)  Petiole_breadth_mm     OK   53307    0.022807   
6   Panicle length (cm)   Panicle_length_cm     OK   53307    0.027732   

      top_chr    top_ps  
1  OX419221.1  51253255  
0  OX419208.1  82291441  
3  OX419236.1    124445  
4  OX419211.1  42638788  
2  OX419236.1    124445  
5  OX419216.1  12599235  
6  OX419221.1  48962949  

Saved: trait_results/gwas_trait_summary.csv


In [25]:
for _, row in summary_df.iterrows():
    if row["status"] != "OK":
        continue

    trait = row["trait"]
    safe_trait = row["safe_trait"]
    assoc_file = f"output/{safe_trait}.assoc.txt"

    gwas = pd.read_csv(assoc_file, sep="\t")
    gwas = gwas.dropna(subset=["p_wald"]).copy()
    gwas = gwas[gwas["p_wald"] > 0].copy()

    if len(gwas) == 0:
        continue

    # ---------- Manhattan ----------
    man = gwas.copy()
    man["chr_num"] = man["chr"].apply(chr_to_num)
    man["minus_log10_p"] = -np.log10(man["p_wald"])
    man = man.sort_values(["chr_num", "ps"]).reset_index(drop=True)

    chrom_sizes = man.groupby("chr_num")["ps"].max().to_dict()

    offset = 0
    offsets = {}
    ticks = []
    ticklabels = []

    for chrom in sorted(man["chr_num"].unique()):
        offsets[chrom] = offset
        chrom_max = chrom_sizes[chrom]
        ticks.append(offset + chrom_max / 2)
        ticklabels.append(str(chrom))
        offset += chrom_max

    man["cum_pos"] = man.apply(lambda r: r["ps"] + offsets[r["chr_num"]], axis=1)

    plt.figure(figsize=(14, 6))
    for chrom in sorted(man["chr_num"].unique()):
        d = man[man["chr_num"] == chrom]
        plt.scatter(d["cum_pos"], d["minus_log10_p"], s=10)

    bonf = 0.05 / len(man)
    plt.axhline(-np.log10(bonf), linestyle="--")
    plt.xlabel("Genomic position")
    plt.ylabel("-log10(p)")
    plt.title(f"Manhattan Plot - {trait}")
    plt.xticks(ticks, ticklabels, rotation=90)
    plt.tight_layout()
    plt.savefig(f"trait_plots/{safe_trait}_manhattan.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ---------- QQ ----------
    pvals = np.sort(gwas["p_wald"].values)
    n = len(pvals)
    expected = -np.log10(np.arange(1, n + 1) / (n + 1))
    observed = -np.log10(pvals)

    plt.figure(figsize=(6, 6))
    plt.scatter(expected, observed, s=12)
    m = max(expected.max(), observed.max())
    plt.plot([0, m], [0, m], linestyle="--")
    plt.xlabel("Expected -log10(p)")
    plt.ylabel("Observed -log10(p)")
    plt.title(f"QQ Plot - {trait}")
    plt.tight_layout()
    plt.savefig(f"trait_plots/{safe_trait}_qq.png", dpi=300, bbox_inches="tight")
    plt.close()

print("Saved all Manhattan and QQ plots.")

Saved all Manhattan and QQ plots.


In [26]:
best_hits = []

for _, row in summary_df.iterrows():
    if row["status"] != "OK":
        continue

    safe_trait = row["safe_trait"]
    trait = row["trait"]
    assoc_file = f"output/{safe_trait}.assoc.txt"

    gwas = pd.read_csv(assoc_file, sep="\t")
    gwas = gwas.dropna(subset=["p_wald"]).copy()

    if len(gwas) == 0:
        continue

    top = gwas.sort_values("p_wald").iloc[0]

    best_hits.append({
        "trait": trait,
        "chr": top["chr"],
        "rs": top["rs"],
        "ps": top["ps"],
        "beta": top["beta"],
        "se": top["se"],
        "p_wald": top["p_wald"],
        "p_lrt": top["p_lrt"] if "p_lrt" in top.index else np.nan,
        "p_score": top["p_score"] if "p_score" in top.index else np.nan
    })

best_hits_df = pd.DataFrame(best_hits)
print(best_hits_df)
best_hits_df.to_csv("trait_results/best_hit_per_trait.csv", index=False)
print("\nSaved: trait_results/best_hit_per_trait.csv")

                  trait         chr rs        ps      beta        se  \
0     Stem Breadth (cm)  OX419221.1  .  51253255 -4.475000  0.190941   
1      Plant Height (m)  OX419208.1  .  82291441  0.681250  0.095129   
2     Leaf breadth (cm)  OX419236.1  .    124445 -2.221276  0.338371   
3   Petiole length (cm)  OX419211.1  .  42638788  3.451870  0.648502   
4      Leaf Length (cm)  OX419236.1  .    124445 -3.050001  0.617387   
5  Petiole breadth (mm)  OX419216.1  .  12599235  0.618930  0.143156   
6   Panicle length (cm)  OX419221.1  .  48962949  9.424510  2.347233   

     p_wald         p_lrt   p_score  
0  0.000170  9.770930e-07  0.112330  
1  0.005608  2.100211e-04  0.119210  
2  0.007190  2.273140e-04  0.119461  
3  0.012955  6.149272e-04  0.123440  
4  0.015908  1.629798e-03  0.129395  
5  0.022807  1.660209e-03  0.129534  
6  0.027732  2.131449e-03  0.131544  

Saved: trait_results/best_hit_per_trait.csv


In [27]:
suggestive_rows = []

for _, row in summary_df.iterrows():
    if row["status"] != "OK":
        continue

    safe_trait = row["safe_trait"]
    trait = row["trait"]
    assoc_file = f"output/{safe_trait}.assoc.txt"

    gwas = pd.read_csv(assoc_file, sep="\t")
    gwas = gwas.dropna(subset=["p_wald"]).copy()

    n_total = len(gwas)
    n_p005 = (gwas["p_wald"] < 0.05).sum()
    n_p001 = (gwas["p_wald"] < 0.01).sum()
    bonf = 0.05 / n_total if n_total > 0 else np.nan
    n_bonf = (gwas["p_wald"] < bonf).sum() if n_total > 0 else 0

    suggestive_rows.append({
        "trait": trait,
        "n_snps": n_total,
        "n_p_lt_0.05": int(n_p005),
        "n_p_lt_0.01": int(n_p001),
        "bonf_threshold": bonf,
        "n_bonf_significant": int(n_bonf)
    })

suggestive_df = pd.DataFrame(suggestive_rows)
print(suggestive_df)
suggestive_df.to_csv("trait_results/trait_significance_summary.csv", index=False)
print("\nSaved: trait_results/trait_significance_summary.csv")

                  trait  n_snps  n_p_lt_0.05  n_p_lt_0.01  bonf_threshold  \
0     Stem Breadth (cm)   53307           20           17    9.379631e-07   
1      Plant Height (m)   53307           18           18    9.379631e-07   
2     Leaf breadth (cm)   53307           13            1    9.379631e-07   
3   Petiole length (cm)   53307         2048            0    9.379631e-07   
4      Leaf Length (cm)   53307            1            0    9.379631e-07   
5  Petiole breadth (mm)   53307         2049            0    9.379631e-07   
6   Panicle length (cm)   53307         2061            0    9.379631e-07   

   n_bonf_significant  
0                   0  
1                   0  
2                   0  
3                   0  
4                   0  
5                   0  
6                   0  

Saved: trait_results/trait_significance_summary.csv


In [28]:
print("=== output/ ===")
!ls -lh output

print("\n=== trait_results/ ===")
!ls -lh trait_results

print("\n=== trait_plots/ ===")
!ls -lh trait_plots

print("\n=== trait_pheno_files/ ===")
!ls -lh trait_pheno_files

=== output/ ===
total 50M
-rw-r--r-- 1 root root  350 Mar 25 05:34 kinship.cXX.txt
-rw-r--r-- 1 root root  875 Mar 25 05:34 kinship.log.txt
-rw-r--r-- 1 root root 7.1M Mar 25 05:38 Leaf_breadth_cm.assoc.txt
-rw-r--r-- 1 root root 1.4K Mar 25 05:38 Leaf_breadth_cm.log.txt
-rw-r--r-- 1 root root 7.1M Mar 25 05:38 Leaf_Length_cm.assoc.txt
-rw-r--r-- 1 root root 1.4K Mar 25 05:38 Leaf_Length_cm.log.txt
-rw-r--r-- 1 root root 7.1M Mar 25 05:39 Panicle_length_cm.assoc.txt
-rw-r--r-- 1 root root 1.4K Mar 25 05:39 Panicle_length_cm.log.txt
-rw-r--r-- 1 root root 7.1M Mar 25 05:38 Petiole_breadth_mm.assoc.txt
-rw-r--r-- 1 root root 1.4K Mar 25 05:38 Petiole_breadth_mm.log.txt
-rw-r--r-- 1 root root 7.1M Mar 25 05:38 Petiole_length_cm.assoc.txt
-rw-r--r-- 1 root root 1.4K Mar 25 05:38 Petiole_length_cm.log.txt
-rw-r--r-- 1 root root 7.1M Mar 25 05:38 Plant_Height_m.assoc.txt
-rw-r--r-- 1 root root 1.4K Mar 25 05:38 Plant_Height_m.log.txt
-rw-r--r-- 1 root root 7.1M Mar 25 05:38 Stem_Breadth_cm.a

In [29]:
import os
import zipfile
from pathlib import Path

zip_name = "GWAS_GEMMA_results_bundle.zip"

items_to_include = [
    "output",
    "trait_results",
    "trait_plots",
    "trait_pheno_files",
    "geno_filtered.bed",
    "geno_filtered.bim",
    "geno_filtered.fam",
    "geno_filtered.log",
    "geno_filtered.nosex",
    "geno.bed",
    "geno.bim",
    "geno.fam",
    "geno.log",
    "geno.nosex",
    "quantitative.xlsx"
]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in items_to_include:
        path = Path(item)
        if not path.exists():
            print(f"Skipping missing: {item}")
            continue

        if path.is_file():
            zf.write(path, arcname=path.name)
        else:
            for subpath in path.rglob("*"):
                if subpath.is_file():
                    zf.write(subpath, arcname=str(subpath))

print(f"Created: {zip_name}")
print(f"Size: {Path(zip_name).stat().st_size / (1024*1024):.2f} MB")

Created: GWAS_GEMMA_results_bundle.zip
Size: 13.59 MB


In [30]:
from google.colab import files
files.download("GWAS_GEMMA_results_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create output folders for manuscript figures
Path("manuscript_figures").mkdir(exist_ok=True)
Path("manuscript_figures/individual").mkdir(exist_ok=True)
Path("manuscript_figures/combined").mkdir(exist_ok=True)

# Global matplotlib settings for publication-quality figures
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 1.0,
    "lines.linewidth": 1.2,
    "pdf.fonttype": 42,   # editable text in Illustrator
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

def safe_name(name: str) -> str:
    import re
    name = name.strip()
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"[\s/]+", "_", name)
    return name

def chr_to_num(x):
    x = str(x)
    try:
        return int(x)
    except:
        x = x.upper()
        if x == "X":
            return 23
        elif x == "Y":
            return 24
        elif x in ["MT", "M"]:
            return 25
        else:
            return 99

In [32]:
summary_df = pd.read_csv("trait_results/gwas_trait_summary.csv")
summary_df = summary_df[summary_df["status"] == "OK"].copy()

print(summary_df)

                  trait          safe_trait status  n_snps  top_p_wald  \
0     Stem Breadth (cm)     Stem_Breadth_cm     OK   53307    0.000170   
1      Plant Height (m)      Plant_Height_m     OK   53307    0.005608   
2     Leaf breadth (cm)     Leaf_breadth_cm     OK   53307    0.007190   
3   Petiole length (cm)   Petiole_length_cm     OK   53307    0.012955   
4      Leaf Length (cm)      Leaf_Length_cm     OK   53307    0.015908   
5  Petiole breadth (mm)  Petiole_breadth_mm     OK   53307    0.022807   
6   Panicle length (cm)   Panicle_length_cm     OK   53307    0.027732   

      top_chr    top_ps  
0  OX419221.1  51253255  
1  OX419208.1  82291441  
2  OX419236.1    124445  
3  OX419211.1  42638788  
4  OX419236.1    124445  
5  OX419216.1  12599235  
6  OX419221.1  48962949  


In [33]:
for _, row in summary_df.iterrows():
    trait = row["trait"]
    safe_trait = row["safe_trait"]
    assoc_file = f"output/{safe_trait}.assoc.txt"

    gwas = pd.read_csv(assoc_file, sep="\t")
    gwas = gwas.dropna(subset=["p_wald"]).copy()
    gwas = gwas[gwas["p_wald"] > 0].copy()

    if len(gwas) == 0:
        continue

    gwas["chr_num"] = gwas["chr"].apply(chr_to_num)
    gwas["minus_log10_p"] = -np.log10(gwas["p_wald"])
    gwas = gwas.sort_values(["chr_num", "ps"]).reset_index(drop=True)

    # cumulative genomic position
    chrom_sizes = gwas.groupby("chr_num")["ps"].max().to_dict()
    offsets = {}
    ticks = []
    ticklabels = []
    offset = 0

    for chrom in sorted(gwas["chr_num"].unique()):
        offsets[chrom] = offset
        chrom_max = chrom_sizes[chrom]
        ticks.append(offset + chrom_max / 2)
        ticklabels.append(str(chrom))
        offset += chrom_max

    gwas["cum_pos"] = gwas.apply(lambda r: r["ps"] + offsets[r["chr_num"]], axis=1)

    # alternate colors by chromosome using default matplotlib cycle
    unique_chroms = sorted(gwas["chr_num"].unique())

    fig, ax = plt.subplots(figsize=(10, 4.8))

    for i, chrom in enumerate(unique_chroms):
        d = gwas[gwas["chr_num"] == chrom]
        ax.scatter(
            d["cum_pos"],
            d["minus_log10_p"],
            s=8,
            alpha=0.85,
            rasterized=True
        )

    bonf = 0.05 / len(gwas)
    suggestive = 1e-4

    ax.axhline(-np.log10(bonf), linestyle="--", linewidth=1.2, label="Bonferroni threshold")
    ax.axhline(-np.log10(suggestive), linestyle=":", linewidth=1.1, label="Suggestive threshold (1e-4)")

    ax.set_xlabel("Genomic position")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(f"Manhattan plot: {trait}")
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticklabels, rotation=90)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, loc="upper right")

    plt.tight_layout()

    base = f"manuscript_figures/individual/{safe_trait}_manhattan"
    plt.savefig(f"{base}.png", bbox_inches="tight")
    plt.savefig(f"{base}.pdf", bbox_inches="tight")
    plt.savefig(f"{base}.svg", bbox_inches="tight")
    plt.close()

print("Saved individual Manhattan plots in PNG, PDF, and SVG.")

Saved individual Manhattan plots in PNG, PDF, and SVG.


In [34]:
for _, row in summary_df.iterrows():
    trait = row["trait"]
    safe_trait = row["safe_trait"]
    assoc_file = f"output/{safe_trait}.assoc.txt"

    gwas = pd.read_csv(assoc_file, sep="\t")
    gwas = gwas.dropna(subset=["p_wald"]).copy()
    gwas = gwas[gwas["p_wald"] > 0].copy()

    if len(gwas) == 0:
        continue

    pvals = np.sort(gwas["p_wald"].values)
    n = len(pvals)

    expected = -np.log10(np.arange(1, n + 1) / (n + 1))
    observed = -np.log10(pvals)

    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    ax.scatter(expected, observed, s=10, alpha=0.8, rasterized=True)

    m = max(expected.max(), observed.max())
    ax.plot([0, m], [0, m], linestyle="--", linewidth=1.2)

    ax.set_xlabel(r"Expected $-\log_{10}(p)$")
    ax.set_ylabel(r"Observed $-\log_{10}(p)$")
    ax.set_title(f"QQ plot: {trait}")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    base = f"manuscript_figures/individual/{safe_trait}_qq"
    plt.savefig(f"{base}.png", bbox_inches="tight")
    plt.savefig(f"{base}.pdf", bbox_inches="tight")
    plt.savefig(f"{base}.svg", bbox_inches="tight")
    plt.close()

print("Saved individual QQ plots in PNG, PDF, and SVG.")

Saved individual QQ plots in PNG, PDF, and SVG.


In [35]:
n_traits = len(summary_df)
ncols = 2
nrows = int(np.ceil(n_traits / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.2 * nrows))
axes = np.array(axes).reshape(-1)

for ax, (_, row) in zip(axes, summary_df.iterrows()):
    trait = row["trait"]
    safe_trait = row["safe_trait"]
    assoc_file = f"output/{safe_trait}.assoc.txt"

    gwas = pd.read_csv(assoc_file, sep="\t")
    gwas = gwas.dropna(subset=["p_wald"]).copy()
    gwas = gwas[gwas["p_wald"] > 0].copy()

    gwas["chr_num"] = gwas["chr"].apply(chr_to_num)
    gwas["minus_log10_p"] = -np.log10(gwas["p_wald"])
    gwas = gwas.sort_values(["chr_num", "ps"]).reset_index(drop=True)

    chrom_sizes = gwas.groupby("chr_num")["ps"].max().to_dict()
    offsets = {}
    ticks = []
    ticklabels = []
    offset = 0

    for chrom in sorted(gwas["chr_num"].unique()):
        offsets[chrom] = offset
        chrom_max = chrom_sizes[chrom]
        ticks.append(offset + chrom_max / 2)
        ticklabels.append(str(chrom))
        offset += chrom_max

    gwas["cum_pos"] = gwas.apply(lambda r: r["ps"] + offsets[r["chr_num"]], axis=1)

    for chrom in sorted(gwas["chr_num"].unique()):
        d = gwas[gwas["chr_num"] == chrom]
        ax.scatter(d["cum_pos"], d["minus_log10_p"], s=6, alpha=0.8, rasterized=True)

    bonf = 0.05 / len(gwas)
    ax.axhline(-np.log10(bonf), linestyle="--", linewidth=1.0)
    ax.set_title(trait)
    ax.set_xlabel("Genomic position")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_xticks(ticks[::max(1, len(ticks)//8)])
    ax.set_xticklabels(ticklabels[::max(1, len(ticklabels)//8)], rotation=90)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# remove unused panels
for j in range(len(summary_df), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

base = "manuscript_figures/combined/all_traits_manhattan_panels"
plt.savefig(f"{base}.png", bbox_inches="tight")
plt.savefig(f"{base}.pdf", bbox_inches="tight")
plt.savefig(f"{base}.svg", bbox_inches="tight")
plt.close()

print("Saved combined multi-panel Manhattan figure.")

Saved combined multi-panel Manhattan figure.


In [36]:
n_traits = len(summary_df)
ncols = 3
nrows = int(np.ceil(n_traits / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
axes = np.array(axes).reshape(-1)

for ax, (_, row) in zip(axes, summary_df.iterrows()):
    trait = row["trait"]
    safe_trait = row["safe_trait"]
    assoc_file = f"output/{safe_trait}.assoc.txt"

    gwas = pd.read_csv(assoc_file, sep="\t")
    gwas = gwas.dropna(subset=["p_wald"]).copy()
    gwas = gwas[gwas["p_wald"] > 0].copy()

    pvals = np.sort(gwas["p_wald"].values)
    n = len(pvals)
    expected = -np.log10(np.arange(1, n + 1) / (n + 1))
    observed = -np.log10(pvals)

    ax.scatter(expected, observed, s=8, alpha=0.8, rasterized=True)
    m = max(expected.max(), observed.max())
    ax.plot([0, m], [0, m], linestyle="--", linewidth=1.0)

    ax.set_title(trait)
    ax.set_xlabel(r"Expected $-\log_{10}(p)$")
    ax.set_ylabel(r"Observed $-\log_{10}(p)$")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for j in range(len(summary_df), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

base = "manuscript_figures/combined/all_traits_qq_panels"
plt.savefig(f"{base}.png", bbox_inches="tight")
plt.savefig(f"{base}.pdf", bbox_inches="tight")
plt.savefig(f"{base}.svg", bbox_inches="tight")
plt.close()

print("Saved combined multi-panel QQ figure.")

Saved combined multi-panel QQ figure.


In [37]:
best_hits_df = pd.read_csv("trait_results/best_hit_per_trait.csv")
best_hits_df["minus_log10_p"] = -np.log10(best_hits_df["p_wald"])

best_hits_df = best_hits_df.sort_values("minus_log10_p", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(best_hits_df["trait"], best_hits_df["minus_log10_p"])

bonf = 0.05 / 53307  # from your run summary
ax.axvline(-np.log10(bonf), linestyle="--", linewidth=1.2, label="Bonferroni threshold")

ax.set_xlabel(r"Top signal per trait: $-\log_{10}(p)$")
ax.set_ylabel("Trait")
ax.set_title("Strongest association signal for each quantitative trait")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)

plt.tight_layout()

base = "manuscript_figures/combined/top_signal_per_trait_barplot"
plt.savefig(f"{base}.png", bbox_inches="tight")
plt.savefig(f"{base}.pdf", bbox_inches="tight")
plt.savefig(f"{base}.svg", bbox_inches="tight")
plt.close()

print("Saved top-signal summary bar plot.")

Saved top-signal summary bar plot.


In [38]:
import zipfile
from pathlib import Path

zip_name = "manuscript_high_quality_figures.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for subpath in Path("manuscript_figures").rglob("*"):
        if subpath.is_file():
            zf.write(subpath, arcname=str(subpath))

print(f"Created: {zip_name}")
print(f"Size: {Path(zip_name).stat().st_size / (1024*1024):.2f} MB")

Created: manuscript_high_quality_figures.zip
Size: 7.31 MB


In [39]:
from google.colab import files
files.download("manuscript_high_quality_figures.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>